In [2]:
import pandas as pd

data = pd.read_csv("../data/unsw_nb15/NF-UNSW-NB15-v3_raw.csv")
data.head()

,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,1424242193040,1424242193043,59.166.0.2,4894,149.171.126.3,53,17,5.0,146,2,...,0,0,0,0,0,0,0,0,0,Benign
1,1424242192744,1424242193079,59.166.0.4,52671,149.171.126.6,31992,6,11.0,4704,28,...,0,91,12,19,0,90,12,19,0,Benign
2,1424242190649,1424242193109,59.166.0.0,47290,149.171.126.9,6881,6,37.0,13662,238,...,0,1843,10,119,0,1843,5,88,0,Benign
3,1424242193145,1424242193146,59.166.0.8,43310,149.171.126.7,53,17,5.0,146,2,...,0,0,0,0,0,0,0,0,0,Benign
4,1424242193239,1424242193241,59.166.0.1,45870,149.171.126.1,53,17,5.0,130,2,...,0,0,0,0,0,0,0,0,0,Benign


 I want to train the model to recognize some structure/pattern in the consecutive flows issued by an ip. Because I know that network data is heavy tailed I assume
 there will be few IPs that issue most of the traffic. If I create training data from this dataset I will have many more sequences containing flows
 from these few heavy talkers which means my dataset would be imbalanced which means that my model would learn the traffic of these particular IPs very well
 but not of the many other ips. Thus, I now want to see if this is the case, is my dataset imbalanced? How many flows are issued by each IP?


In [3]:
flows_per_ip = data.groupby("IPV4_SRC_ADDR").size().sort_values(ascending=False)
print(flows_per_ip.describe())
total_flows = flows_per_ip.sum()
print("total number of flows: ", total_flows)
print(flows_per_ip)
print(flows_per_ip.quantile([0.5, 0.75, 0.9, 0.95, 0.99]))

# how many flows are sent by the top 10 percent IPs?
flows_per_ip
print("percentage: ", (flows_per_ip.quantile([0.9]) / total_flows) * 100)

count        40.000000
mean      59135.600000
std       94856.796931
min           1.000000
25%         352.250000
50%         653.500000
75%       79923.250000
max      225316.000000
dtype: float64
total number of flows:  2365424
IPV4_SRC_ADDR
59.166.0.2         225316
59.166.0.4         224942
59.166.0.1         224766
59.166.0.0         224274
59.166.0.3         223608
59.166.0.5         222835
59.166.0.9         215681
59.166.0.8         215238
59.166.0.7         214339
59.166.0.6         214165
175.45.176.1        35176
175.45.176.3        33820
175.45.176.0        31442
175.45.176.2        27255
149.171.126.18      11317
149.171.126.12       3940
149.171.126.14       3848
149.171.126.15       3833
149.171.126.10       3580
10.40.85.1            665
10.40.182.1           642
10.40.85.10           479
149.171.126.5         446
149.171.126.8         431
149.171.126.4         420
149.171.126.2         413
149.171.126.3         401
149.171.126.1         390
149.171.126.6         354
1

In [4]:
perc = 0.05
n_ips = int(len(flows_per_ip) * perc)
nr_flows_top_n_ips = flows_per_ip.head(n_ips).sum()
percentage_top_n_ips = (nr_flows_top_n_ips / total_flows) * 100
print(f"{perc * 100} % of the ips make up {percentage_top_n_ips} percent of flows")

5.0 % of the ips make up 19.034980620810476 percent of flows


In [5]:
# what does the top sender contribute:
flows_per_ip.iloc[0] / flows_per_ip.sum() * 100

np.float64(9.525395869831371)